# E0006 — vLLM dependency probe (NO GPU)

Purpose: resolve the offline runtime dependency problem **without spending L4 quota**.

Required settings:
- Accelerator: **None**
- Internet: **ON**

This notebook performs a pip **dry run only**. It does not install vLLM, load the Nemotron model, or submit anything.

Expected output: `/kaggle/working/e0006_vllm_dependency_probe.json`


In [ ]:
from __future__ import annotations

import importlib.metadata
import json
import platform
import subprocess
import sys
import tempfile
import urllib.request
from pathlib import Path

OUT = Path('/kaggle/working/e0006_vllm_dependency_probe.json')
REPORT = Path('/kaggle/working/e0006_vllm_pip_dry_run_report.json')
VLLM_VERSION = '0.27.1'

def version(name):
    try:
        return importlib.metadata.version(name)
    except Exception:
        return None

base = {
    'python': sys.version,
    'platform': platform.platform(),
    'packages': {name: version(name) for name in [
        'torch', 'transformers', 'accelerate', 'triton',
        'flashinfer-python', 'vllm', 'safetensors'
    ]},
}

# Source-of-truth release metadata and exact CUDA requirements for v0.27.1.
release_url = 'https://api.github.com/repos/vllm-project/vllm/releases/tags/v0.27.1'
req_url = 'https://raw.githubusercontent.com/vllm-project/vllm/v0.27.1/requirements/cuda.txt'
with urllib.request.urlopen(release_url, timeout=30) as r:
    release = json.load(r)
with urllib.request.urlopen(req_url, timeout=30) as r:
    cuda_requirements = r.read().decode('utf-8')

assets = []
for a in release.get('assets', []):
    name = a.get('name', '')
    if 'x86_64' in name and (name.endswith('.whl') or name.endswith('.tar.gz')):
        assets.append({
            'name': name,
            'bytes': a.get('size'),
            'sha256': (a.get('digest') or '').removeprefix('sha256:'),
            'url': a.get('browser_download_url'),
        })

cmd = [
    sys.executable, '-m', 'pip', 'install',
    '--dry-run', '--report', str(REPORT),
    f'vllm[flashinfer]=={VLLM_VERSION}',
]
cp = subprocess.run(cmd, capture_output=True, text=True, timeout=900)

pip_report = None
if REPORT.exists():
    pip_report = json.loads(REPORT.read_text(encoding='utf-8'))

planned = []
if isinstance(pip_report, dict):
    for item in pip_report.get('install', []):
        meta = item.get('metadata') or {}
        dl = item.get('download_info') or {}
        planned.append({
            'name': meta.get('name'),
            'version': meta.get('version'),
            'requested': item.get('requested'),
            'url': dl.get('url'),
        })

critical_names = {'torch', 'triton', 'transformers', 'flashinfer-python', 'vllm', 'torchvision', 'torchaudio'}
critical_plan = [x for x in planned if str(x.get('name', '')).lower() in critical_names]

payload = {
    'experiment': 'E0006',
    'gate': 'D0_DEPENDENCY_DRY_RUN_NO_GPU',
    'base_environment': base,
    'vllm_version': VLLM_VERSION,
    'vllm_cuda_requirements_txt': cuda_requirements,
    'release_assets_x86_64': assets,
    'pip_command': cmd,
    'pip_returncode': cp.returncode,
    'pip_stdout_tail': cp.stdout[-12000:],
    'pip_stderr_tail': cp.stderr[-12000:],
    'planned_install_count': len(planned),
    'critical_planned_changes': critical_plan,
    'planned_installs': planned,
    'decision_rule': (
        'Do not run L4 Gate B from the base image. Review this report first and build an isolated offline runtime.'
    ),
}
OUT.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps({
    'base_environment': base,
    'pip_returncode': cp.returncode,
    'planned_install_count': len(planned),
    'critical_planned_changes': critical_plan,
    'release_assets_x86_64': assets,
}, indent=2, sort_keys=True))
print(f'\nWROTE: {OUT}')
